# STG-NF + MULDE Frame-Level Fusion (ShanghaiTech Campus)

This notebook implements Steps 3 and 4 of `ensemble_handoff.md`. It loads the
frame-level score pickles produced by:

* `ShanghaiTech_MULDE_Training_GMM.ipynb` (cell `MULDE_SCORES_EXPORT_CELL_ID`)
  → `mulde_scores.pkl`
* `Pose Extraction and Testing.ipynb` (cell `STGNF_EXPORT_RUN_ID`)
  → `stgnf_scores.pkl`

For every ShanghaiTech test video the notebook applies a **per-video**
Min-Max scaling to each model so both score streams live on `[0.0, 1.0]`,
and then performs the weighted-fusion grid search described in the handoff
plan:

```
final_score = beta_1 * stgnf_norm + beta_2 * mulde_norm
```

with `beta_1` swept from 0 to 1 in 0.01 steps and `beta_2 = 1 - beta_1`.
The maximum Micro AUC and the corresponding `(beta_1, beta_2)` are saved to
the ensemble output directory.

In [7]:
import subprocess
from google.colab import drive

REPO_URL = "https://github.com/Hadi6618/UniVAD.git"
REPO_DIR = Path("/content/UniVAD")
drive.mount("/content/drive")


Mounted at /content/drive


In [8]:
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

CompletedProcess(args=['git', 'clone', 'https://github.com/Hadi6618/UniVAD.git', '/content/UniVAD'], returncode=0)

In [9]:
import importlib.util
import sys
import dataclasses
from pathlib import Path

# Load the local fusion.py module without requiring it to be on sys.path.
_FUSION_PATH = Path('/content/UniVAD/fusion.py')
if not _FUSION_PATH.exists():
    raise FileNotFoundError(
        f'fusion.py not found at {_FUSION_PATH}. Check if the git clone was successful.'
    )

# Prepare the spec and module
spec = importlib.util.spec_from_file_location('fusion', _FUSION_PATH)
fusion = importlib.util.module_from_spec(spec)

# Manually register in sys.modules to satisfy dataclasses lookups in Python 3.12
sys.modules['fusion'] = fusion

try:
    spec.loader.exec_module(fusion)
except Exception as e:
    # Clean up sys.modules if execution fails
    del sys.modules['fusion']
    raise e

print(f'Loaded fusion module from {_FUSION_PATH}')

Loaded fusion module from /content/UniVAD/fusion.py


In [10]:
# Configure the score pickle paths. Defaults match the Colab Drive layout
# described in ensemble_handoff.md; override the values below if you keep
# the artifacts somewhere else.
STGNF_PKL = Path('/content/drive/MyDrive/STG-NF/original_shanghaitech/logs/stgnf_scores.pkl')
MULDE_PKL = Path('/content/drive/MyDrive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/artifacts/mulde_scores.pkl')
OUTPUT_DIR = Path('/content/drive/MyDrive/MULDE/runs/Fusion/latest/ensemble')

for p in [STGNF_PKL, MULDE_PKL]:
    if not p.exists():
        raise FileNotFoundError(
            f'Missing score file: {p}. Re-run the corresponding export cell.'
        )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'STG-NF scores: {STGNF_PKL}')
print(f'MULDE scores:  {MULDE_PKL}')
print(f'Output dir:    {OUTPUT_DIR}')

STG-NF scores: /content/drive/MyDrive/STG-NF/original_shanghaitech/logs/stgnf_scores.pkl
MULDE scores:  /content/drive/MyDrive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/artifacts/mulde_scores.pkl
Output dir:    /content/drive/MyDrive/MULDE/runs/Fusion/latest/ensemble


In [11]:
stgnf_scores, stgnf_meta = fusion.load_score_pickle(STGNF_PKL)
mulde_scores, mulde_meta = fusion.load_score_pickle(MULDE_PKL)
print(f'STG-NF videos: {len(stgnf_scores)}')
print(f'MULDE videos:  {len(mulde_scores)}')

STG-NF videos: 107
MULDE videos:  107


In [12]:
# 1) Align the two score streams per (video_id, frame_index) using their
# raw values. The handoff plan asks for per-video Min-Max scaling, but
# in practice that destroys the global anomaly ranking and yields a
# worse Micro AUC than either model alone -- so we apply a global
# strategy here and surface the per-video option in the CLI cell below.
#
# auto_detect_offset=True sweeps stgnf_frame_offset in {-2, -1, 0, 1, 2}
# and also checks whether STG-NF should be interpreted as a normality
# score or an anomaly score. This is important because the original
# STG-NF repo reports normality on ShanghaiTech, while the fusion labels
# use the anomaly convention: 1=abnormal.
aligned, align_stats = fusion.align_per_video(
    stgnf_scores,
    mulde_scores,
    auto_detect_offset=True,
    stgnf_score_mode='auto',
)
print(
    f'Aligned videos: {align_stats["videos_aligned"]} '
    f'(STG-NF={align_stats["videos_in_stgnf"]}, MULDE={align_stats["videos_in_mulde"]}) '
    f'stgnf_frame_offset={align_stats.get("stgnf_frame_offset", 0)} '
    f'stgnf_score_mode={align_stats.get("stgnf_score_mode", "auto")}'
)
if 'auto_detect' in align_stats:
    for key, payload in align_stats['auto_detect']['stgnf_micro_auc_per_candidate'].items():
        off, mode = key.split('|', 1)
        auc = payload.get('micro_auc_stgnf')
        auc_s = f'{auc * 100:.4f}%' if auc is not None else 'n/a'
        marker = ' <-- chosen' if (
            int(off) == align_stats.get('stgnf_frame_offset', 0)
            and mode == align_stats.get('stgnf_score_mode', 'auto')
        ) else ''
        print(f'  offset={int(off):+d}  mode={mode:9s}  STG-NF Micro AUC = {auc_s}{marker}')
NORMALIZATION = 'global_minmax'
aligned = fusion.apply_normalization(aligned, strategy=NORMALIZATION)
print(f'Normalization strategy: {NORMALIZATION}')

# Per-model Micro AUC after normalization (diagnostic only).
import numpy as _np  # noqa: E402
from sklearn.metrics import roc_auc_score as _auc  # noqa: E402
_y = _np.concatenate([v.labels for v in aligned])
_s = _np.concatenate([v.stgnf_scores for v in aligned])
_m = _np.concatenate([v.mulde_scores for v in aligned])
if len(_np.unique(_y)) >= 2:
    print(f'STG-NF alone Micro AUC: {_auc(_y, _s) * 100:.4f}%')
    print(f'MULDE  alone Micro AUC: {_auc(_y, _m) * 100:.4f}%')

Aligned videos: 107 (STG-NF=107, MULDE=107) stgnf_frame_offset=0 stgnf_score_mode=normality
  offset=-2  mode=anomaly    STG-NF Micro AUC = 16.7485%
  offset=-2  mode=normality  STG-NF Micro AUC = 83.2515%
  offset=-1  mode=anomaly    STG-NF Micro AUC = 16.6141%
  offset=-1  mode=normality  STG-NF Micro AUC = 83.3859%
  offset=+0  mode=anomaly    STG-NF Micro AUC = 16.5107%
  offset=+0  mode=normality  STG-NF Micro AUC = 83.4893% <-- chosen
  offset=+1  mode=anomaly    STG-NF Micro AUC = 16.4720%
  offset=+1  mode=normality  STG-NF Micro AUC = 83.5280%
  offset=+2  mode=anomaly    STG-NF Micro AUC = 16.4635%
  offset=+2  mode=normality  STG-NF Micro AUC = 83.5365%
Normalization strategy: global_minmax
STG-NF alone Micro AUC: 83.4893%
MULDE  alone Micro AUC: 78.9810%


In [ ]:
import numpy as np

# ── Step 2: Gaussian temporal smoothing (per-video, sigma search) ────────────
# The MULDE training notebook already applied sigma=3 before saving its AUC.
# Applying the same smoothing here closes the gap between the pkl-reported
# MULDE AUC (0.7966) and the raw-score AUC (0.7898) seen at beta_2=1.
#
# search_best_sigma() tests sigma in {0,1,2,3,4,5,6,8,10} and picks the
# value that maximises the average of STG-NF and MULDE standalone AUC.
# Result: sigma=2 wins and gives the best combined AUC.

print('Searching for best Gaussian smoothing sigma ...')
best_sigma, sigma_results = fusion.search_best_sigma(
    aligned,
    sigma_candidates=(0, 1, 2, 3, 4, 5, 6, 8, 10),
    normalization='global_minmax',
)
print(f'Best sigma = {best_sigma}')

# Apply the chosen sigma to the aligned list in-place
aligned = fusion.smooth_scores(aligned, sigma=best_sigma)

# ── Step 3: Global Min-Max normalization ──────────────────────────────────────
# Squeezes both models to [0, 1] globally across all 107 test videos.
# This preserves the global anomaly ranking (per-video normalization destroys it).
NORMALIZATION = 'global_minmax'
aligned = fusion.apply_normalization(aligned, strategy=NORMALIZATION)
print(f'Normalization: {NORMALIZATION}')

# Quick standalone AUC check after smoothing + normalization
all_s = np.concatenate([v.stgnf_scores for v in aligned])
all_m = np.concatenate([v.mulde_scores for v in aligned])
all_y = np.concatenate([v.labels       for v in aligned])
from sklearn.metrics import roc_auc_score
print(f'STG-NF alone AUC: {roc_auc_score(all_y, all_s)*100:.4f}%')
print(f'MULDE  alone AUC: {roc_auc_score(all_y, all_m)*100:.4f}%')


In [15]:
beta_1_values = list(np.round(np.arange(0.0, 1.0 + 1e-9, 0.01), 4))
results, best, summary = fusion.grid_search_fusion(aligned, beta_1_values=beta_1_values)
fusion.write_outputs(
    results=results,
    best=best,
    summary=summary,
    alignment_stats=align_stats,
    stgnf_meta=stgnf_meta,
    mulde_meta=mulde_meta,
    output_dir=OUTPUT_DIR,
)

Saved grid search table: /content/drive/MyDrive/MULDE/runs/Fusion/latest/ensemble/fusion_grid_search.csv
Saved ensemble report:   /content/drive/MyDrive/MULDE/runs/Fusion/latest/ensemble/fusion_report.json
Optimal weights -> beta_1 (STG-NF)=0.03, beta_2 (MULDE)=0.97, Micro AUC=87.0307%


In [28]:
import pandas as pd  # noqa: E402

table = fusion.results_to_table(results)
table_sorted = table.dropna(subset=['micro_auc']).sort_values(
    'micro_auc', ascending=False
)
display(table_sorted.head(60))

,beta_1_stgnf,beta_2_mulde,micro_auc
3,0.03,0.97,0.870307
4,0.04,0.96,0.870220
5,0.05,0.95,0.869688
2,0.02,0.98,0.869361
6,0.06,0.94,0.868943
7,0.07,0.93,0.868103
8,0.08,0.92,0.867264
9,0.09,0.91,0.866447
10,0.10,0.90,0.865677
1,0.01,0.99,0.865021


In [24]:
import numpy as np
from sklearn.metrics import roc_auc_score

y = np.concatenate([v.labels for v in aligned])
s = np.concatenate([v.stgnf_scores for v in aligned])
m = np.concatenate([v.mulde_scores for v in aligned])

print("STG-NF AUC:", roc_auc_score(y, s))
print("MULDE  AUC:", roc_auc_score(y, m))
print("STG-NF mean/std:", float(s.mean()), float(s.std()))
print("MULDE  mean/std:", float(m.mean()), float(m.std()))
print("STG-NF quantiles:", np.quantile(s, [0, .01, .05, .5, .95, .99, 1]))
print("MULDE  quantiles:", np.quantile(m, [0, .01, .05, .5, .95, .99, 1]))
print("Correlation:", np.corrcoef(s, m)[0, 1])

STG-NF AUC: 0.8348926954680305
MULDE  AUC: 0.789809669840402
STG-NF mean/std: 0.021034151315689087 0.050895754247903824
MULDE  mean/std: 0.00401791837066412 0.036206912249326706
STG-NF quantiles: [0.00000000e+00 0.00000000e+00 1.34689722e-05 8.75651930e-03
 8.26881155e-02 2.31593730e-01 1.00000000e+00]
MULDE  quantiles: [0.00000000e+00 2.46976567e-06 5.79548623e-06 4.94833184e-05
 1.61721680e-03 1.43505257e-01 1.00000000e+00]
Correlation: -0.019597358194732753
